**Instalação e Importação de Pacotes**

In [ ]:
!pip install pandas
!pip install numpy
!pip install matplotlib
!pip install scikit-learn
!pip install xgboost

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

**Início do Processo**

In [ ]:
#Importação da base

dados = pd.read_excel("base_final2.xlsx")

In [ ]:
dados.head()

In [ ]:
dados.info()

In [ ]:
categorical_columns = ['Segmento',
'Estado_Civil',
'Etnia',
'Genero',
'Geracao',
'Grau_de_Instrucao',
'Demitido_ou_Ativo']

for col in categorical_columns:
  dados[col] = dados[col].astype('category')

In [ ]:
# Dummização das variáveis categóricas

dados = pd.get_dummies(dados,
                       columns=[
                                'Demitido_ou_Ativo'],
                       drop_first=True,
                       dtype='int')

In [ ]:
# Separação das variáveis X e da variável Y

X = dados.drop(columns=['Demitido_ou_Ativo_Demitido'])
y = dados['Demitido_ou_Ativo_Demitido']

In [ ]:
# Separação entre treino e teste das variáveis X e da variável Y

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.3,
                                                    random_state=100)

In [ ]:
# Algorítmo de classificação XGBoost

# Parâmetros do grid - Alterar para gerar novos resultados

# n_estimators: qtde de árvores no modelo
# max_depth: profundidade máxima das árvores
# colsample_bytree: percentual de variáveis X subamostradas para cada árvore
# learning_rate: taxa de aprendizagem

#param_grid_xgb = {
 #   'n_estimators': [100, 300, 600, 900],
 #   'max_depth': [3, 6, 9, 12],
 #   'colsample_bytree': [0.25, 0.5, 0.75, 1],
 #   'learning_rate': [0.01, 0.1, 0.15, 0.2]
#}
# Melhores parâmetros
# param_grid_xgb = {
# colsample_bytree': 0.25,
# 'learning_rate': 0.01,
# 'max_depth': 9,
# 'n_estimators': 300

param_grid_xgb = {
    'n_estimators': [300],
    'max_depth': [9],
    'colsample_bytree': [0.25],
    'learning_rate': [0.01]
}

# Identificar o algoritmo em uso
xgb_grid = XGBClassifier(enable_categorical=True, random_state=100)

# Treinar os modelos para o grid search
xgb_grid_model = GridSearchCV(estimator = xgb_grid,
                              param_grid = param_grid_xgb,
                              scoring='accuracy',           # Avaliador da Matriz de Confusão
                              cv=5,                         # Cross-Validation
                              verbose=2                     # Mostra as iterações
                              )

xgb_grid_model.fit(X_train, y_train)

xgb_grid_model.best_params_

In [ ]:
xgb_best = xgb_grid_model.best_estimator_

xgb_features = pd.DataFrame({'features':X.columns.tolist(),
                            'importance':xgb_best.feature_importances_})

print(xgb_features)

importances = xgb_features.to_csv('importances.csv', sep=',', encoding='utf-8', index=False)

In [ ]:
#%% Obtendo os valores preditos pelo XGBoost

# Predict na base de treinamento
xgb_grid_pred_train_class = xgb_best.predict(X_train)
xgb_grid_pred_train_prob = xgb_best.predict_proba(X_train)

# Predict na base de testes
xgb_grid_pred_test_class = xgb_best.predict(X_test)
xgb_grid_pred_test_prob = xgb_best.predict_proba(X_test)


In [ ]:
#%% Matriz de confusão (base de treino)

xgb_cm_train = confusion_matrix(xgb_grid_pred_train_class, y_train)
cm_xgb_train = ConfusionMatrixDisplay(xgb_cm_train)

plt.rcParams['figure.dpi'] = 600
cm_xgb_train.plot(colorbar=False, cmap='summer')
plt.title('XGBoost: Treino')
plt.xlabel('Observado (Real)')
plt.ylabel('Classificado (Modelo)')
plt.show()

acc_xgb_train = accuracy_score(y_train, xgb_grid_pred_train_class)
sens_xgb_train = recall_score(y_train, xgb_grid_pred_train_class, pos_label=1)
espec_xgb_train = recall_score(y_train, xgb_grid_pred_train_class, pos_label=0)
prec_xgb_train = precision_score(y_train, xgb_grid_pred_train_class)

print("Avaliação do XGBoost (Base de Treino)")
print(f"Acurácia: {acc_xgb_train:.1%}")
print(f"Sensibilidade: {sens_xgb_train:.1%}")
print(f"Especificidade: {espec_xgb_train:.1%}")
print(f"Precision: {prec_xgb_train:.1%}")



In [ ]:
#%% Matriz de confusão (base de teste)

xgb_cm_test = confusion_matrix(xgb_grid_pred_test_class, y_test)
cm_xgb_test = ConfusionMatrixDisplay(xgb_cm_test)

plt.rcParams['figure.dpi'] = 300
cm_xgb_test.plot(colorbar=False, cmap='summer')
plt.title('XGBoost: Teste')
plt.xlabel('Observado (Real)')
plt.ylabel('Classificado (Modelo)')
plt.show()

acc_xgb_test = accuracy_score(y_test, xgb_grid_pred_test_class)
sens_xgb_test = recall_score(y_test, xgb_grid_pred_test_class, pos_label=1)
espec_xgb_test = recall_score(y_test, xgb_grid_pred_test_class, pos_label=0)
prec_xgb_test = precision_score(y_test, xgb_grid_pred_test_class)

print("Avaliação do XGBoost (Base de Teste)")
print(f"Acurácia: {acc_xgb_test:.1%}")
print(f"Sensibilidade: {sens_xgb_test:.1%}")
print(f"Especificidade: {espec_xgb_test:.1%}")
print(f"Precision: {prec_xgb_test:.1%}")


In [ ]:
#%% Curva ROC (base de teste)

# Parametrizando a função da curva ROC (real vs. previsto)
fpr_xgb, tpr_xgb, thresholds_xgb = roc_curve(y_test, xgb_grid_pred_test_prob[:,1])
roc_auc_xgb = auc(fpr_xgb, tpr_xgb)

# Plotando a curva ROC
plt.figure(figsize=(15,10), dpi=600)
plt.plot(fpr_xgb, tpr_xgb, color='green', linewidth=4)
plt.plot(fpr_xgb, fpr_xgb, color='gray', linestyle='dashed')
plt.title('AUC-ROC XGBoost: %g' % round(roc_auc_xgb, 3), fontsize=22)
plt.xlabel('1 - Especificidade', fontsize=20)
plt.ylabel('Sensibilidade', fontsize=20)
plt.xticks(np.arange(0, 1.1, 0.2), fontsize=14)
plt.yticks(np.arange(0, 1.1, 0.2), fontsize=14)
plt.show()

In [ ]:
categorical_columns = ['Segmento',
'Estado_Civil',
'Etnia',
'Genero',
'Geracao',
'Grau_de_Instrucao',
'Demitido_ou_Ativo']

for col in categorical_columns:
  dados[col] = dados[col].astype('category')

In [ ]:
#%% Realizando previsões para observações de fora da amostra

novo_cliente = pd.DataFrame({'Segmento': ['Empresas_Privadas'],
                            'Estado_Civil': ['Solteiro'],
                            'Etnia': ['Branco'],
                            'Genero': ['Masculino'],
                            'Geracao': ['Geracao_Y'],
                            'Grau_de_Instrucao': ['2_Grau_Completo'],
                            'Faltas': [5],
                            'Salario': [1890],
                            'Idade': [32],
                            'Horas_Trabalhadas': [220],
                            'Afastamentos': [0],
                            'Atestados_Entregues': [0],
                            'Quantidade_de_Filhos': [0],
                            'Possui_Plano_de_Saude': [0],
                            'Conte_Comigo': [10],
                            'Ajuda_de_Custo': [0],
                            'Indique_um_Amigo': [0],
                            'Vale_Farmacia': [1],
                            'Bonus_Efetivacao': [1],
                            'Vale_Oculos': [0],
                             })

# Convertendo as colunas categóricas para o tipo 'category'
categorical_cols_for_prediction = ['Segmento', 'Estado_Civil', 'Etnia', 'Genero', 'Geracao', 'Grau_de_Instrucao']
for col in categorical_cols_for_prediction:
    novo_cliente[col] = novo_cliente[col].astype('category')

cliente_xgb = xgb_best.predict_proba(novo_cliente)
print(cliente_xgb)